# 01. 수집 (외부데이터 정제/가공)
식약처 + 농정원 수집. app.rag 재사용. 산출물 data/lab/01_raw.json

In [ ]:
import sys,os,asyncio,json
from pathlib import Path
B=(Path.cwd().parent/'backend') if Path.cwd().name=='notebooks' else Path.cwd()/'backend'
B=B.resolve(); sys.path.insert(0,str(B)); os.chdir(B)
try: sys.stdout.reconfigure(encoding='utf-8')
except Exception: pass
from dotenv import load_dotenv; load_dotenv()
print('backend:',B,'| EMBED_PROVIDER:',os.getenv('EMBED_PROVIDER'))

In [ ]:
from app.rag.data.fetch_cookrcp import fetch_cookrcp
from app.rag.data.fetch_mafra import fetch_mafra
# 주의: fetch_* 는 이미 normalize_*(embed_text 포함)까지 수행해 반환 — 01_raw 는 명칭만 raw, 실제 정규화 완료 상태
cook=asyncio.run(fetch_cookrcp(limit=20, api_key=os.getenv('FOOD_SAFETY_API_KEY')))
mafra=asyncio.run(fetch_mafra(limit=10))
raw=cook+mafra
print('collected:',len(raw),'(cookrcp',len(cook),'+ mafra',len(mafra),')')
if not raw:
    raise SystemExit('수집 0건 — FOOD_SAFETY_API_KEY 필요 / mafra 는 호출IP 제한(로컬은 보통 0). 키 확인 후 재실행.')
lab=B/'data'/'lab'; lab.mkdir(parents=True,exist_ok=True)
(lab/'01_raw.json').write_text(json.dumps(raw,ensure_ascii=False),encoding='utf-8')
print('saved -> data/lab/01_raw.json | 예:',raw[0]['name'])